In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import re
import glob
import json
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages

#==================================================================================================

PLOT_TITLE    = 'Benchmark for Geant4-DNA (IRT) / Weak-scaling'
PDF_FILE_NAME = 'performance-plot.pdf'

SIM_DATA = [
    {'dir': 'IRT-11.1.0', 'label': 'G4DNA-11.1.0 AlmaLinux 9.1 (native, gcc 11.3.1)'},
    {'dir': 'IRT-11.0.3', 'label': 'G4DNA-11.0.3 AlmaLinux 9.1 (native, gcc 11.3.1)'},
]

#==================================================================================================

DIRPREFIX = 'sim_'

MAX_THREAD_NUM = 70
MAX_THROUGHPUT = 80000

WORKDIR = os.getcwd()

#--------------------------------------------------------------------------------------------------
def get_filelist():
    
    dirlist = glob.glob(DIRPREFIX + '*')
    dirlist.sort(key=lambda s: int(re.search(r'\d+', s).group()))
    
    threads = []
    for x in dirlist:
        threads.append(x.split(DIRPREFIX)[1])
    
    filelist = []
    for x in threads:
        filename = os.path.join(f'{DIRPREFIX}{x}', f'benchmark_{x}.json')
        filelist.append(filename)
    
    return filelist

#--------------------------------------------------------------------------------------------------
def make_data_frame(path):
    os.chdir(path)
    
    filelist = get_filelist()
    
    thread_num = []
    eps = []
    for x in filelist:
        f = open(x, 'r')
        js = json.load(f)
        thread_num.append(js['summary']['thread_number'])
        eps.append(js['summary']['throughput'])
    
    df = pd.DataFrame(data=dict(ThreadNum=thread_num, EPS=eps))
  
    os.chdir(WORKDIR)
  
    return df

#--------------------------------------------------------------------------------------------------
def gvalue(path, name):
    os.chdir(path)
    
    filelist = get_filelist()
    
    thread_num = []
    gval_start = []
    gval_end = []
    
    for x in filelist:
        f = open(x, 'r')
        js = json.load(f)
        thread_num.append(js['summary']['thread_number'])
        gval_start.append(js['summary']['gvalue'][name]['chem_start'])
        gval_end.append(js['summary']['gvalue'][name]['chem_end'])
        
    df = pd.DataFrame(data=dict(ThreadNum=thread_num, GValStart=gval_start, GValEnd=gval_end))
    
    os.chdir(WORKDIR)
    
    return df

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))
plt.title(PLOT_TITLE, fontsize=16)

for x in SIM_DATA:
    df = make_data_frame(x['dir'])
    plt.plot(df['ThreadNum'], df['EPS'], marker='o', linewidth=3.0, label=x['label'])

ax.ticklabel_format(style='sci', axis='y', scilimits=(0,0))
plt.xlabel("Thread Number", fontsize=16)
plt.ylabel("#Histories/min", fontsize=16)
plt.xlim(0, MAX_THREAD_NUM)
plt.ylim(0, MAX_THROUGHPUT)
plt.tick_params(labelsize=16)
plt.minorticks_on()
plt.legend(fontsize=12)
plt.grid(linestyle='dotted')
plt.plot()

# save as a PDF file
pdf = PdfPages(PDF_FILE_NAME)
pdf.savefig(fig)

In [ ]:
fig = plt.figure(figsize=(8,6))
plt.title("Chemistry Regression (OH radicals at 1 ps)", fontsize=16)

for x in SIM_DATA:
    df = gvalue(x['dir'], 'hydroxyl_radical')
    plt.plot(df['ThreadNum'], df['GValStart'], marker='o', linewidth=3.0, label=x['label'])
    
plt.xlabel("Thread Number", fontsize=16)
plt.ylabel("G-value", fontsize=16)
plt.xlim(0, MAX_THREAD_NUM)
plt.ylim(4, 6)
plt.tick_params(labelsize=16)
plt.minorticks_on()
plt.legend(fontsize=12)
plt.grid(linestyle='dotted')
plt.plot()

pdf.savefig(fig)

In [ ]:
fig = plt.figure(figsize=(8,6))
plt.title("Chemistry Regression (OH radicals at 1 us)", fontsize=16)

for x in SIM_DATA:
    df = gvalue(x['dir'], 'hydroxyl_radical')
    plt.plot(df['ThreadNum'], df['GValEnd'], marker='o', linewidth=3.0, label=x['label'])
    
plt.xlabel("Thread Number", fontsize=16)
plt.ylabel("G-value", fontsize=16)
plt.xlim(0, MAX_THREAD_NUM)
plt.ylim(2.6, 3.2)
plt.tick_params(labelsize=16)
plt.minorticks_on()
plt.legend(fontsize=12)
plt.grid(linestyle='dotted')
plt.plot()

pdf.savefig(fig)

In [ ]:
fig = plt.figure(figsize=(8,6))
plt.title("Chemistry Regression (Solvated Electrons at 1 ps)", fontsize=16)

for x in SIM_DATA:
    df = gvalue(x['dir'], 'solvated_electron')
    plt.plot(df['ThreadNum'], df['GValStart'], marker='o', linewidth=3.0, label=x['label'])
    
plt.xlabel("Thread Number", fontsize=16)
plt.ylabel("G-value", fontsize=16)
plt.xlim(0, MAX_THREAD_NUM)
plt.ylim(3, 5)
plt.tick_params(labelsize=16)
plt.minorticks_on()
plt.legend(fontsize=12)
plt.grid(linestyle='dotted')
plt.plot()

pdf.savefig(fig)

In [ ]:
fig = plt.figure(figsize=(8,6))
plt.title("Chemistry Regression (Solvated Electrons at 1 us)", fontsize=16)

for x in SIM_DATA:
    df = gvalue(x['dir'], 'solvated_electron')
    plt.plot(df['ThreadNum'], df['GValEnd'], marker='o', linewidth=3.0, label=x['label'])
    
plt.xlabel("Thread Number", fontsize=16)
plt.ylabel("G-value", fontsize=16)
plt.xlim(0, MAX_THREAD_NUM)
plt.ylim(2, 4)
plt.tick_params(labelsize=16)
plt.minorticks_on()
plt.legend(fontsize=12)
plt.grid(linestyle='dotted')
plt.plot()

pdf.savefig(fig)

In [ ]:
fig = plt.figure(figsize=(8,6))
plt.title("Chemistry Regression (H2O2 molecules at 1 ps)", fontsize=16)

for x in SIM_DATA:
    df = gvalue(x['dir'], 'H2O2')
    plt.plot(df['ThreadNum'], df['GValStart'], marker='o', linewidth=3.0, label=x['label'])
    
plt.xlabel("Thread Number", fontsize=16)
plt.ylabel("G-value", fontsize=16)
plt.xlim(0, MAX_THREAD_NUM)
plt.ylim(0, 0.1)
plt.tick_params(labelsize=16)
plt.minorticks_on()
plt.legend(fontsize=12)
plt.grid(linestyle='dotted')
plt.plot()

pdf.savefig(fig)

In [ ]:
fig = plt.figure(figsize=(8,6))
plt.title("Chemistry Regression (H2O2 molecules at 1 us)", fontsize=16)

for x in SIM_DATA:
    df = gvalue(x['dir'], 'H2O2')
    plt.plot(df['ThreadNum'], df['GValEnd'], marker='o', linewidth=3.0, label=x['label'])
    
plt.xlabel("Thread Number", fontsize=16)
plt.ylabel("G-value", fontsize=16)
plt.xlim(0, MAX_THREAD_NUM)
plt.ylim(0.5, 0.7)
plt.tick_params(labelsize=16)
plt.minorticks_on()
plt.legend(fontsize=12)
plt.grid(linestyle='dotted')
plt.plot()

pdf.savefig(fig)
pdf.close()